In [0]:
%sql
create or refresh streaming table starboy_9.m_hop.stream_bronze 
as 
select * from stream read_files('/Volumes/starboy_9/m_hop/v_files/data_files/', format => 'json', multiLine => 'true')

In [0]:
create or refresh streaming table starboy_9.m_hop.stream_silver
as 
select customer_id,cast(order_date AS DATE) as order_date,order_id,payment_method,product_name,quantity,shipping_address.city as city,shipping_address.zip_code as zip_code,
    status,total_amount
 from stream(starboy_9.m_hop.stream_bronze)
where customer_id is not null and customer_id like 'CUST%' and cast(order_date AS date) is not null and order_id is not null and payment_method is not null and payment_method in ('UPI','COD','CREDIT_CARD','DEBIT_CARD') and product_name is not null and quantity is not null and status is not null and status in ('DELIVERED','PENDING','CANCELLED','SHIPPED') and total_amount is not null and shipping_address.city is not null and shipping_address.zip_code is not null

In [0]:
create or refresh streaming table starboy_9.m_hop.stream_gold1
as 
select payment_method,count(*) as total_no_of_orders from stream(starboy_9.m_hop.stream_silver) group by payment_method order by 2 desc

In [0]:
create or refresh streaming table starboy_9.m_hop.stream_gold2
as 
select order_id,round(sum(total_amount /quantity),2) as product_price from stream(starboy_9.m_hop.stream_silver) group by order_id 

In [0]:
create or refresh streaming table starboy_9.m_hop.stream_gold3
as 
select city,count(*) as orders_per_city from stream(starboy_9.m_hop.stream_silver) group by city order by 2 desc

In [0]:
create or refresh streaming table starboy_9.m_hop.stream_gold4
as 
select * from stream(starboy_9.m_hop.stream_silver) where status='DELIVERED'